# 🤖 RAG Chatbot — Gemini 2.5 Flash + BM25 Hybrid Retrieval
**Hệ thống hỏi đáp PDF** · Dense + BM25 + RRF Hybrid · Evaluation Metrics · Gradio UI

## 📦 BƯỚC 1: Cài đặt thư viện
Chạy ô này, sau đó **Runtime → Restart Session**, rồi mới chạy Bước 2.

In [1]:
!pip install -q langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters
!pip install -q pypdf gradio rank_bm25 pysqlite3-binary sentence-transformers
print('✅ CÀI ĐẶT HOÀN TẤT! Hãy vào Runtime → Restart Session trước khi chạy Bước 2.')


✅ CÀI ĐẶT HOÀN TẤT! Hãy vào Runtime → Restart Session trước khi chạy Bước 2.


## ⚙️ BƯỚC 2: Khởi tạo hệ thống RAG
Lấy Gemini API Key miễn phí tại: https://aistudio.google.com/app/apikey

Upload file PDF lên Colab (panel trái → biểu tượng 📁 → Upload) trước khi chạy.

In [11]:
import os, sys, warnings, time, hashlib
warnings.filterwarnings('ignore')

# ── 1. API Key ────────────────────────────────────────────────────────────────
from getpass import getpass
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass('🔑 Nhập Gemini API Key: ')

# ── 2. Vá lỗi SQLite (giữ lại phòng khi môi trường vẫn có pysqlite3) ─────────
try:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass

# ── 3. Import thư viện ────────────────────────────────────────────────────────
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI          # chỉ giữ LLM
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field, PrivateAttr                            # FIX: thêm PrivateAttr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from typing import List
import torch

# ── 4. Nạp PDF ────────────────────────────────────────────────────────────────
print('📂 Đang nạp PDF từ /content/ ...')
loader = DirectoryLoader('/content/', glob='./*.pdf', loader_cls=PyPDFLoader, show_progress=True)
raw_data = loader.load()

if not raw_data:
    print('⚠️  Không tìm thấy file PDF. Hãy upload trước!')
else:
    print(f'✅ Đã nạp {len(raw_data)} trang từ {len(set(d.metadata["source"] for d in raw_data))} file.')

    # ── 5. Tiền xử lý ────────────────────────────────────────────────────────
    data = []
    for doc in raw_data:
        page_num = doc.metadata.get('page', 0)
        if 0 <= page_num < 50:
            clean_lines = [line.strip() for line in doc.page_content.splitlines() if line.strip()]
            doc.page_content = '\n'.join(clean_lines)
            data.append(doc)
    print(f'🧹 Sau lọc trang 1–50: còn {len(data)} trang.')

    # ── 6. Chunking ───────────────────────────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(data)
    for c in chunks:
        c.metadata['source'] = os.path.basename(c.metadata.get('source', 'unknown'))
        c.metadata['page']   = c.metadata.get('page', 0) + 1
    print(f'✂️  Đã chia thành {len(chunks)} chunks (chunk_size=1000, overlap=100).')
        # ── 6.5. Tự động trích xuất viết tắt từ corpus ───────────────────────────
    import re

    def build_abbrev_map(chunks):
        abbrev_map = {}
        p1 = re.compile(
            r'([A-Z][a-zA-Z]+(?:[\s\n]+[A-Z][a-zA-Z]+){1,5})'
            r'[\s\n]*\(([A-Z]{2,8})\)',
            re.MULTILINE
        )
        p2 = re.compile(
            r'([A-Z]{2,}(?:[\s\n]+[A-Z]{2,}){1,5})'
            r'[\s\n]*\(([A-Z]{2,8})\)',
            re.MULTILINE
        )
        p3 = re.compile(
            r'\b([A-Z]{2,8})\b'
            r'[\s\n]*\(([A-Z][a-zA-Z]+(?:[\s\n]+[A-Z][a-zA-Z]+){1,5})\)',
            re.MULTILINE
        )
        for chunk in chunks:
            text = chunk.page_content
            for match in p1.finditer(text):
                abbrev = match.group(2).lower().strip()
                full   = match.group(1).lower().strip().replace('\n', ' ')
                abbrev_map[abbrev] = full
            for match in p2.finditer(text):
                abbrev = match.group(2).lower().strip()
                full   = match.group(1).lower().strip().replace('\n', ' ')
                if abbrev not in abbrev_map:
                    abbrev_map[abbrev] = full
            for match in p3.finditer(text):
                abbrev = match.group(1).lower().strip()
                full   = match.group(2).lower().strip().replace('\n', ' ')
                if abbrev not in abbrev_map:
                    abbrev_map[abbrev] = full
        return abbrev_map

    def expand_query(query: str) -> str:
        """Thay viết tắt trong query bằng dạng đầy đủ trước khi truyền vào retriever."""
        query_lower = query.lower()
        tokens = query.lower().split()
        return " ".join([ABBREV_MAP.get(tok, tok) for tok in tokens])

    ABBREV_MAP = build_abbrev_map(chunks)
    print(f"✅ Tìm được {len(ABBREV_MAP)} viết tắt:")
    for k, v in ABBREV_MAP.items():
        print(f"   '{k}' → '{v}'")

    # ── 7. TF-IDF Retriever (thay thế Gemini Embedding + Chroma) ─────────────
    class TfidfRetriever(BaseRetriever):
        """
        Custom LangChain-compatible retriever dùng TF-IDF + Cosine Similarity.
        Kế thừa BaseRetriever → .invoke(query) hoạt động chuẩn LangChain.
        """
        chunks: List[Document] = Field(description="Danh sách Document sau khi chunk")
        k: int                 = Field(default=5, description="Số document trả về")

        # FIX: dùng PrivateAttr() thay vì gán = None trực tiếp
        _vectorizer:   TfidfVectorizer = PrivateAttr(default=None)
        _tfidf_matrix: np.ndarray      = PrivateAttr(default=None)

        def model_post_init(self, __context):
            """Fit TF-IDF ngay sau khi khởi tạo object."""
            corpus = [doc.page_content for doc in self.chunks]
            self._vectorizer = TfidfVectorizer(
                ngram_range=(1, 2),  # unigram + bigram
                min_df=1,
                max_df=0.95,
                sublinear_tf=True,   # dùng log(tf) thay tf thô
            )
            self._tfidf_matrix = self._vectorizer.fit_transform(corpus)

        def _get_relevant_documents(self, query: str) -> List[Document]:
            """Phương thức bắt buộc — được .invoke() của LangChain gọi nội bộ."""
            query_vec = self._vectorizer.transform([query])
            scores    = cosine_similarity(query_vec, self._tfidf_matrix).flatten()
            top_k_idx = np.argsort(scores)[::-1][:self.k]
            return [self.chunks[i] for i in top_k_idx]

        def invoke_with_score(self, query: str) -> List[tuple]:
            """Trả về (Document, score) — dùng cho get_retriever_docs(mode='dense')."""
            query_vec = self._vectorizer.transform([query])
            scores    = cosine_similarity(query_vec, self._tfidf_matrix).flatten()
            top_k_idx = np.argsort(scores)[::-1][:self.k]
            return [(self.chunks[i], round(float(scores[i]), 4)) for i in top_k_idx]

    print("⚙️  Đang fit TF-IDF trên toàn bộ corpus chunks...")
    dense_retriever = TfidfRetriever(chunks=chunks, k=5)
    print(f"✅ TF-IDF Retriever sẵn sàng!")
    print(f"   • Vocab size : {len(dense_retriever._vectorizer.vocabulary_):,} terms")
    print(f"   • Matrix     : {dense_retriever._tfidf_matrix.shape}  (chunks × terms)")
    print(f"   • Top-k      : {dense_retriever.k}")

    # ── 8. BM25 Retriever ─────────────────────────────────────────────────────
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 5

    # ── 8.5. SBERT Dense Retriever (Vietnamese Bi-Encoder) ──────────────────
    from sentence_transformers import SentenceTransformer

    SBERT_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
    print(f"⏳ Đang tải SBERT model: {SBERT_MODEL_NAME} ...")
    _sbert_model_global = SentenceTransformer(SBERT_MODEL_NAME)
    _sbert_model_global.eval()
    print(f"✅ SBERT model loaded!")
    print(f"   • Embedding dim: {_sbert_model_global.get_sentence_embedding_dimension()}")
    print(f"   • Device: {_sbert_model_global.device}")

    class SbertDenseRetriever(BaseRetriever):
        """Dense Vector Retriever dùng SentenceTransformer (SBERT) tiếng Việt."""
        chunks: List[Document] = Field(description="Danh sách Document sau khi chunk")
        k: int = Field(default=5, description="Số document trả về")
        _model: object = PrivateAttr(default=None)
        _embeddings: np.ndarray = PrivateAttr(default=None)

        def model_post_init(self, __context):
            print("⏳ SbertDenseRetriever: đang encode chunks...")
            self._model = _sbert_model_global
            corpus = [doc.page_content for doc in self.chunks]
            with torch.no_grad():
                self._embeddings = self._model.encode(
                    corpus, batch_size=64,
                    convert_to_numpy=True, show_progress_bar=True
                )
            print(f"✅ SBERT Retriever sẵn sàng!")
            print(f"   • Chunks: {len(self.chunks):,}")
            print(f"   • Embedding shape: {self._embeddings.shape}")

        def _get_relevant_documents(self, query: str) -> List[Document]:
            with torch.no_grad():
                q_emb = self._model.encode(query, convert_to_numpy=True)
            scores = cosine_similarity([q_emb], self._embeddings)[0]
            top_k_idx = np.argsort(scores)[::-1][:self.k]
            return [self.chunks[i] for i in top_k_idx]

        def invoke_with_score(self, query: str) -> List[tuple]:
            with torch.no_grad():
                q_emb = self._model.encode(query, convert_to_numpy=True)
            scores = cosine_similarity([q_emb], self._embeddings)[0]
            top_k_idx = np.argsort(scores)[::-1][:self.k]
            return [(self.chunks[i], round(float(scores[i]), 4)) for i in top_k_idx]

    print("⚙️  Đang khởi tạo SbertDenseRetriever...")
    sbert_retriever = SbertDenseRetriever(chunks=chunks, k=5)


    # ── 9. RRF Hybrid + Re-ranking ────────────────────────────────────────────
    def _doc_key(doc):
        raw = f"{doc.metadata.get('source','')}|{doc.metadata.get('page','')}|{doc.page_content[:200]}"
        return hashlib.md5(raw.encode()).hexdigest()

    def rrf_hybrid(query, k=5, rrf_k=60):
        bm25_docs  = bm25_retriever.invoke(query)
        dense_docs = dense_retriever.invoke(query)   # gọi TfidfRetriever.invoke()

        scores = {}
        for rank, doc in enumerate(bm25_docs, start=1):
            key = _doc_key(doc)
            if key not in scores:
                scores[key] = {'score': 0.0, 'doc': doc}
            scores[key]['score'] += 1 / (rrf_k + rank)

        for rank, doc in enumerate(dense_docs, start=1):
            key = _doc_key(doc)
            if key not in scores:
                scores[key] = {'score': 0.0, 'doc': doc}
            scores[key]['score'] += 1 / (rrf_k + rank)

        ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)
        return [(item['doc'], round(item['score'], 6)) for item in ranked[:k]]

    def rrf_hybrid_sbert(query, k=5, rrf_k=60):
        """Hybrid RRF kết hợp BM25 + SBERT (thay TF-IDF dense)."""
        bm25_docs  = bm25_retriever.invoke(query)
        sbert_docs = sbert_retriever.invoke(query)
        scores = {}
        for rank, doc in enumerate(bm25_docs, start=1):
            key = _doc_key(doc)
            if key not in scores:
                scores[key] = {'score': 0.0, 'doc': doc}
            scores[key]['score'] += 1 / (rrf_k + rank)
        for rank, doc in enumerate(sbert_docs, start=1):
            key = _doc_key(doc)
            if key not in scores:
                scores[key] = {'score': 0.0, 'doc': doc}
            scores[key]['score'] += 1 / (rrf_k + rank)
        ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)
        return [(item['doc'], round(item['score'], 6)) for item in ranked[:k]]

    def rrf_hybrid_tfidf_sbert(query, k=5, c=60):
      global dense_retriever, sbert_retriever
      tfidf_results = dense_retriever.invoke_with_score(query)
      tfidf_docs = [doc for doc, _ in tfidf_results]

      sbert_results = sbert_retriever.invoke_with_score(query)
      sbert_docs = [doc for doc, _ in sbert_results]

      rrf_scores = {}

      for rank, doc in enumerate(tfidf_docs):
        doc_id = doc.page_content
        rrf_scores[doc_id] = rrf_scores.get(doc_id, {'doc': doc, 'score': 0.0})
        rrf_scores[doc_id]['score'] += 1.0 / (c + rank + 1)

      for rank, doc in enumerate(sbert_docs):
        doc_id = doc.page_content
        rrf_scores[doc_id] = rrf_scores.get(doc_id, {'doc': doc, 'score': 0.0})
        rrf_scores[doc_id]['score'] += 1.0 / (c + rank + 1)

      sorted_docs = sorted(rrf_scores.values(), key=lambda x: x['score'], reverse=True)
      return [(item['doc'], round(item['score'], 4)) for item in sorted_docs[:k]]

    def rerank(query, doc_score_pairs, top_n=3):
        import re
        query_words = set(re.findall(r'\w+', query.lower()))
        if not query_words:
            return doc_score_pairs[:top_n]
        reranked = []
        for doc, rrf_score in doc_score_pairs:
            content_words = set(re.findall(r'\w+', doc.page_content.lower()))
            overlap = len(query_words & content_words) / (len(query_words) + 1e-9)
            final_score = round(0.7 * rrf_score * 100 + 0.3 * overlap, 6)
            reranked.append((doc, final_score))
        reranked.sort(key=lambda x: x[1], reverse=True)
        return reranked[:top_n]

    def get_retriever_docs(query, mode='hybrid', k=5):
        if not mode:
            mode = 'hybrid'
        expanded = expand_query(query)
        global dense_retriever, sbert_retriever, bm25_retriever
        if mode == 'bm25':
            docs = bm25_retriever.invoke(expanded)
            return [(d, round(1/(i+1), 4)) for i, d in enumerate(docs[:k])]
        elif mode == 'tfidf':
            results = dense_retriever.invoke_with_score(expanded)
            return [(d, round(float(s), 4)) for d, s in results]
        elif mode == 'sbert':
            results = sbert_retriever.invoke_with_score(expanded)
            return [(d, round(float(s), 4)) for d, s in results]
        elif mode == 'hybrid_tfidf':
            hybrid_tfidf_results = rrf_hybrid_tfidf_sbert(expanded, k=k+2)
            return rerank(expanded, hybrid_tfidf_results, top_n=k)
        else:  # hybrid: BM25 + SBERT via RRF
            hybrid_results = rrf_hybrid_sbert(expanded, k=k+2)
            return rerank(expanded, hybrid_results, top_n=k)

    # ── 10. LLM ───────────────────────────────────────────────────────────────
    llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)

    # ── 11. Prompt ────────────────────────────────────────────────────────────
    SYSTEM_PROMPT = """Bạn là **PTIT Knowledge Assistant** — một chuyên gia phân tích tài liệu học thuật của trường Học viện Công nghệ Bưu chính Viễn thông (PTIT).

## VAI TRÒ & NGUYÊN TẮC
- Bạn CHỈ trả lời dựa trên các đoạn tài liệu được cung cấp trong `[CONTEXT]`.
- Bạn KHÔNG suy diễn, KHÔNG thêm thông tin từ kiến thức nền ngoài tài liệu.
- Mọi thông tin đưa ra phải có trích dẫn số trang rõ ràng.

## QUY TẮC XỬ LÝ KHI KHÔNG TÌM THẤY THÔNG TIN
Nếu `[CONTEXT]` không chứa thông tin liên quan đến câu hỏi, hãy trả lời theo mẫu:
>  **Tài liệu không đề cập đến vấn đề này.**
> Nội dung được cung cấp không đủ để trả lời câu hỏi: *"{{question}}"*. Vui lòng tham khảo thêm tài liệu khác hoặc đặt câu hỏi khác.

## ĐỊNH DẠNG TRẢ LỜI (BẮT BUỘC)
Luôn trả lời theo cấu trúc Markdown sau:

###  Trả lời
[Nội dung trả lời chính — dùng bullet points nếu có nhiều ý, dùng bảng nếu cần so sánh]

###  Trích dẫn nguồn
- **[Tên file]**, Trang **[số trang]**: *[trích dẫn ngắn gọn câu liên quan từ tài liệu]*

---
*Thông tin được tổng hợp từ tài liệu nội bộ PTIT.*"""

    PROMPT = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "[CONTEXT]\n{context}\n\n[CÂU HỎI]\n{question}")
    ])

    def format_docs(doc_score_pairs):
        parts = []
        for d, score in doc_score_pairs:
            src  = d.metadata.get('source', 'unknown')
            page = d.metadata.get('page', '?')
            parts.append(f'[File: {src} | Trang: {page} | Score: {score}]\n{d.page_content}')
        return '\n\n---\n\n'.join(parts)

    def build_rag_chain(mode='hybrid'):
        return (
            {'context': RunnableLambda(lambda q: format_docs(get_retriever_docs(q, mode=mode))),
             'question': RunnablePassthrough()}
            | PROMPT | llm | StrOutputParser()
        )

    rag_chain = build_rag_chain('hybrid')

    print('\n🚀 HỆ THỐNG RAG SẴN SÀNG!')
    print(f'   • LLM        : gemini-2.5-flash')
    print(f'   • Embeddings : TF-IDF (scikit-learn) — chạy cục bộ')  # FIX: cập nhật log
    print(f'   • Chunking   : size=1000, overlap=100')
    print(f'   • Retrievers : BM25 | TF-IDF | SBERT Dense | Hybrid RRF (BM25+SBERT)')
    print(f'   • Chunks     : {len(chunks)}')
    all_chunks = list(chunks)



📂 Đang nạp PDF từ /content/ ...


100%|██████████| 3/3 [00:00<00:00,  6.21it/s]


✅ Đã nạp 59 trang từ 3 file.
🧹 Sau lọc trang 1–50: còn 59 trang.
✂️  Đã chia thành 59 chunks (chunk_size=1000, overlap=100).
✅ Tìm được 0 viết tắt:
⚙️  Đang fit TF-IDF trên toàn bộ corpus chunks...
✅ TF-IDF Retriever sẵn sàng!
   • Vocab size : 2,621 terms
   • Matrix     : (59, 2621)  (chunks × terms)
   • Top-k      : 5
⏳ Đang tải SBERT model: bkai-foundation-models/vietnamese-bi-encoder ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ SBERT model loaded!
   • Embedding dim: 768
   • Device: cuda:0
⚙️  Đang khởi tạo SbertDenseRetriever...
⏳ SbertDenseRetriever: đang encode chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ SBERT Retriever sẵn sàng!
   • Chunks: 59
   • Embedding shape: (59, 768)

🚀 HỆ THỐNG RAG SẴN SÀNG!
   • LLM        : gemini-2.5-flash
   • Embeddings : TF-IDF (scikit-learn) — chạy cục bộ
   • Chunking   : size=1000, overlap=100
   • Retrievers : BM25 | TF-IDF | SBERT Dense | Hybrid RRF (BM25+SBERT)
   • Chunks     : 59


##  BƯỚC 3: Thước đo đánh giá (Evaluation Metrics)
Tính **Exact Match (EM)**, **F1 Score** và **Mean Reciprocal Rank (MRR)** theo giáo trình.

In [ ]:
import re, string, time
from collections import Counter

# ── 1. Tiền xử lý text ────────────────────────────────────────────────────────

def extract_answer_body(text: str) -> str:
    """
    Cắt phần nội dung nằm giữa '### Trả lời' và '### Trích dẫn nguồn'.
    Nếu không tìm thấy pattern → trả về nguyên text.
    """
    # Lấy phần sau "### Trả lời" (hoặc "###  Trả lời" có dấu cách thừa)
    m = re.search(r'#{1,3}\s*Trả lời\s*\n(.*?)(?=#{1,3}\s*Trích dẫn|$)',
                  text, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return text.strip()

def normalize(text: str) -> str:
    """Lowercase, bỏ dấu câu, chuẩn hoá khoảng trắng."""
    text = extract_answer_body(str(text))
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

# ── 2. Exact Match ────────────────────────────────────────────────────────────

def exact_match(prediction: str, ground_truth: str) -> int:

    body_pred = extract_answer_body(prediction)
    clean_pred = normalize(body_pred if len(body_pred) > 10 else prediction)
    clean_gt   = normalize(ground_truth)
    return int(clean_pred == clean_gt)

# ── 3. F1 Score (token-level) ─────────────────────────────────────────────────

def f1_score(prediction: str, ground_truth: str) -> float:
    body_pred = extract_answer_body(prediction)
    pred_text = body_pred if len(body_pred) > 10 else prediction

    pred_tokens  = normalize(pred_text).split()
    truth_tokens = normalize(ground_truth).split()

    if not pred_tokens or not truth_tokens:
        return 0.0
    common      = Counter(pred_tokens) & Counter(truth_tokens)
    num_common  = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

# ── 4. MRR — đối chiếu theo số trang ─────────────────────────────────────────

def mean_reciprocal_rank(retrieved_lists: list, relevant_pages_list: list) -> float:
    """
    retrieved_lists      : list of list[Document] — docs trả về mỗi query
    relevant_pages_list  : list of list[int/str]  — trang chuẩn mỗi query
                           (đọc từ cột 'Trang_Chuan' trong Excel)
    """
    rr_scores = []
    for retrieved_docs, relevant_pages in zip(retrieved_lists, relevant_pages_list):
        # Chuẩn hoá relevant_pages về set[int]
        rel_set = set()
        for p in relevant_pages:
            try:
                rel_set.add(int(p))
            except (ValueError, TypeError):
                pass

        rr = 0.0
        for rank, doc in enumerate(retrieved_docs, start=1):
            doc_page = doc.metadata.get('page', None)
            try:
                if int(doc_page) in rel_set:
                    rr = 1.0 / rank
                    break          # lấy rank đầu tiên khớp
            except (ValueError, TypeError):
                continue
        rr_scores.append(rr)

    return round(sum(rr_scores) / len(rr_scores), 4) if rr_scores else 0.0

# ── 5. Hàm đánh giá tổng hợp ─────────────────────────────────────────────────

def evaluate_retrievers(test_set: list, modes=None, k: int = 5) -> 'pd.DataFrame':
    """
    Chạy đánh giá trên nhiều chế độ retrieval.

    test_set : list[dict] với keys:
        - 'question'         : str
        - 'ground_truth'     : str
        - 'relevant_pages'   : list[int]  ← từ cột 'Trang_Chuan'
    modes    : list chế độ cần đánh giá, mặc định cả 4
    """
    import pandas as pd

    if modes is None:
        modes = ['bm25', 'tfidf', 'sbert', 'hybrid']

    results = {'Mode': [], 'Avg EM': [], 'Avg F1': [], 'MRR': []}

    for mode in modes:
        chain = build_rag_chain(mode)
        em_scores, f1_scores = [], []
        retrieved_lists, relevant_pages_list = [], []

        print(f'\n{"="*60}')
        print(f'🔍 Đang đánh giá: {mode.upper()}')
        print(f'{"="*60}')

        for i, item in enumerate(test_set):
            q           = item['question']
            gt          = item['ground_truth']
            rel_pages   = item.get('relevant_pages', [])

            # ── Gọi LLM ───────────────────────────────────────────────────────
            try:
                pred = chain.invoke(q)
                time.sleep(4)          # Tránh rate-limit Gemini free (15 RPM)
            except Exception as e:
                pred = ''
                print(f'  ⚠️  Q{i+1} lỗi LLM: {e}')
                time.sleep(15)

            em  = exact_match(pred, gt)
            f1  = f1_score(pred, gt)
            em_scores.append(em)
            f1_scores.append(f1)

            # ── Lấy docs cho MRR ──────────────────────────────────────────────
            doc_score_pairs = get_retriever_docs(q, mode=mode, k=k)
            retrieved_docs  = [d for d, _ in doc_score_pairs]
            retrieved_lists.append(retrieved_docs)
            relevant_pages_list.append(rel_pages)

            print(f'  [{i+1:02d}/{len(test_set)}] EM={em} | F1={f1:.4f} | Pages_retrieved={[d.metadata.get("page") for d in retrieved_docs[:3]]} | Pages_expected={rel_pages}')

        # Nghỉ giữa các mode
        print(f'✅ Xong {mode.upper()}. Chờ 10s...')
        time.sleep(10)

        mrr = mean_reciprocal_rank(retrieved_lists, relevant_pages_list)
        results['Mode'].append(mode.upper())
        results['Avg EM'].append(round(sum(em_scores) / len(em_scores), 4))
        results['Avg F1'].append(round(sum(f1_scores) / len(f1_scores), 4))
        results['MRR'].append(mrr)

    df = pd.DataFrame(results).set_index('Mode')
    return df

print('✅ Evaluation Metrics sẵn sàng.')
print('   • exact_match  : so sánh sau khi strip Markdown header')
print('   • f1_score     : token-level trên nội dung trả lời thuần')
print('   • MRR          : đối chiếu doc.metadata["page"] với Trang_Chuan')

✅ Evaluation Metrics sẵn sàng.
   • exact_match  : so sánh sau khi strip Markdown header
   • f1_score     : token-level trên nội dung trả lời thuần
   • MRR          : đối chiếu doc.metadata["page"] với Trang_Chuan


## BƯỚC 4: Chạy đánh giá
Điền câu hỏi và đáp án mẫu vào `TEST_SET` bên dưới.

In [ ]:
import pandas as pd

# ── Đọc dataset ───────────────────────────────────────────────────────────────
df_excel = pd.read_excel('Dataset_Demo_3_Cau_Chien_Thuat.xlsx')

# Kiểm tra tên cột
print("Các cột trong file Excel:", df_excel.columns.tolist())

# ── Build TEST_SET ────────────────────────────────────────────────────────────
TEST_SET = []
for _, row in df_excel.iterrows():
    # 'Trang_Chuan' có thể chứa 1 số (44) hoặc nhiều số cách nhau dấu phẩy (44,46)
    raw_pages = str(row.get('Trang_Chuan', ''))
    rel_pages = []
    for p in raw_pages.split(','):
        p = p.strip()
        if p.isdigit():
            rel_pages.append(int(p))

    TEST_SET.append({
        'question'      : row['Câu hỏi'],
        'ground_truth'  : row['Đáp án chuẩn'],
        'relevant_pages': rel_pages,
    })

print(f'✅ Đã nạp {len(TEST_SET)} câu hỏi vào TEST_SET.')
print(f'\nMẫu câu hỏi đầu tiên:')
print(f'  Q   : {TEST_SET[0]["question"]}')
print(f'  GT  : {TEST_SET[0]["ground_truth"][:80]}...')
print(f'  Pages: {TEST_SET[0]["relevant_pages"]}')

# ── Chạy đánh giá ─────────────────────────────────────────────────────────────
# Nếu muốn chạy nhanh 1 mode trước để test:
# df_result = evaluate_retrievers(TEST_SET, modes=['hybrid'], k=5)

# Chạy đầy đủ cả 4 mode:
df_result = evaluate_retrievers(TEST_SET, modes=['bm25', 'tfidf', 'sbert', 'hybrid'], k=5)

# ── Hiển thị kết quả ──────────────────────────────────────────────────────────
print('\n' + '='*50)
print('📊 KẾT QUẢ ĐÁNH GIÁ')
print('='*50)
print(df_result.to_string())

# Highlight mode tốt nhất từng metric
print('\n🏆 Tốt nhất:')
print(f'  Avg EM  → {df_result["Avg EM"].idxmax()}  ({df_result["Avg EM"].max():.4f})')
print(f'  Avg F1  → {df_result["Avg F1"].idxmax()}  ({df_result["Avg F1"].max():.4f})')
print(f'  MRR     → {df_result["MRR"].idxmax()}     ({df_result["MRR"].max():.4f})')


Các cột trong file Excel: ['STT', 'Câu hỏi', 'Đáp án chuẩn', 'Trang_Chuan', 'Ghi chú']
✅ Đã nạp 3 câu hỏi vào TEST_SET.

Mẫu câu hỏi đầu tiên:
  Q   : Hệ thống Question Answering (QA) được phân loại dựa trên những tiêu chí nào?
  GT  : Hệ thống QA được phân loại dựa trên hai tiêu chí chính: một là nguồn thông tin m...
  Pages: [4]

🔍 Đang đánh giá: BM25
  [01/3] EM=0 | F1=0.3333 | Pages_retrieved=[3, 10, 20] | Pages_expected=[4]
  [02/3] EM=0 | F1=0.5000 | Pages_retrieved=[3, 31, 43] | Pages_expected=[43]
  [03/3] EM=0 | F1=0.1990 | Pages_retrieved=[44, 46, 45] | Pages_expected=[44]
✅ Xong BM25. Chờ 10s...

🔍 Đang đánh giá: TFIDF
  [01/3] EM=0 | F1=0.3220 | Pages_retrieved=[2, 1, 41] | Pages_expected=[4]
  [02/3] EM=0 | F1=0.4606 | Pages_retrieved=[2, 43, 1] | Pages_expected=[43]
  [03/3] EM=0 | F1=0.2647 | Pages_retrieved=[44, 46, 45] | Pages_expected=[44]
✅ Xong TFIDF. Chờ 10s...

🔍 Đang đánh giá: SBERT
  [01/3] EM=0 | F1=0.1687 | Pages_retrieved=[9, 7, 20] | Pages_expected=[4]
  [02

## BƯỚC 5: Giao diện Gradio nâng cao
Chọn phương thức truy xuất, xem câu trả lời + nguồn trích dẫn + điểm xếp hạng.

In [12]:
# =============================================================================
# PATCH: Tính năng "Tải lên tài liệu mới" cho RAG_Gemini_v3.ipynb
#
# HƯỚNG DẪN TÍCH HỢP:
#   - PHẦN A → Thêm vào cuối ô "BƯỚC 2" (sau dòng print '🚀 HỆ THỐNG RAG...')
#   - PHẦN B → Thay thế toàn bộ ô "BƯỚC 5" (cell Gradio hiện tại)
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# PHẦN A — Thêm vào cuối ô BƯỚC 2
# Mục đích: khai báo all_chunks là master list, dùng chung với bm25_retriever
# ─────────────────────────────────────────────────────────────────────────────

# Đặt sau dòng: print(f'   • Chunks     : {len(chunks)}')
all_chunks = list(chunks)   # Master list — sẽ được extend mỗi khi upload


# ─────────────────────────────────────────────────────────────────────────────
# PHẦN B — Thay thế toàn bộ ô BƯỚC 5
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr

# ── CSS: giữ nguyên CSS gốc, thêm style cho upload panel ────────────────────
CUSTOM_CSS = """
.gradio-container { background: #0f1117 !important; }
.message.user {
    background: #1a3a5c !important; color: #e8f4fd !important;
    border-radius: 18px 18px 4px 18px !important;
    margin-left: auto !important; max-width: 75% !important;
    padding: 10px 15px !important; border: 1px solid #2a5a8c !important;
}
.message.bot {
    background: #1e2130 !important; color: #d4d8e8 !important;
    border-radius: 18px 18px 18px 4px !important;
    margin-right: auto !important; max-width: 85% !important;
    padding: 10px 15px !important; border: 1px solid #2e3250 !important;
}
.stat-card { background:#141720; border:1px solid rgba(0,200,255,0.25);
    border-radius:12px; padding:12px 16px; margin-bottom:10px;
    box-shadow:0 0 12px rgba(0,200,255,0.07); }
.stat-card .label { color:#7a8099; font-size:11px; text-transform:uppercase; letter-spacing:1px; }
.stat-card .value { color:#e2e8f0; font-size:14px; font-weight:600; margin-top:4px; }
.stat-card .value.green { color:#4ade80; } .stat-card .value.blue { color:#60a5fa; }
.source-tag { display:inline-block; background:rgba(0,200,255,0.1);
    border:1px solid rgba(0,200,255,0.3); border-radius:20px;
    padding:3px 12px; margin:3px 4px; font-size:12px; color:#7dd3fc; font-weight:500; }
.score-badge { display:inline-block; background:rgba(74,222,128,0.1);
    border:1px solid rgba(74,222,128,0.3); border-radius:8px;
    padding:2px 8px; margin-left:6px; font-size:11px; color:#4ade80; }
.sources-wrapper { padding:10px 4px 4px 4px; border-top:1px solid #2e3250; margin-top:6px; }
.sources-label { font-size:11px; color:#5a6080; text-transform:uppercase;
    letter-spacing:1px; margin-bottom:6px; }
.rounded-input textarea { border-radius:24px !important; padding:12px 20px !important;
    background:#1a1d2e !important; border:1px solid #2e3250 !important;
    color:#e2e8f0 !important; font-size:14px !important; }
.rounded-input textarea:focus { border-color:rgba(0,200,255,0.5) !important;
    box-shadow:0 0 0 3px rgba(0,200,255,0.1) !important; }
.send-btn { border-radius:24px !important; }
.chatbot-wrap .label-wrap { display:none !important; }
.chatbot-wrap { background:#0f1117 !important; border:1px solid #1e2130 !important;
    border-radius:16px !important; }
input[type=range] { accent-color:#00c8ff; }
h1.main-title { background:linear-gradient(90deg,#60a5fa,#00c8ff);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent;
    font-size:26px !important; font-weight:700 !important; }

/* ── Upload panel ── */
.upload-status-ok  { color:#4ade80; font-size:13px; padding:6px 0; }
.upload-status-err { color:#f87171; font-size:13px; padding:6px 0; }
"""

MODE_LABELS = {'BM25 (Keyword)': 'bm25', 'TF-IDF (Sparse)': 'tfidf', 'Dense SBERT (Semantic)': 'sbert', 'Hybrid RRF(BM25 + SBERT)': 'hybrid', 'Hybrid RRF (TFIDF + SBERT)': 'hybrid_tfidf'}

# ── Hàm build source HTML (giữ nguyên) ───────────────────────────────────────
def build_source_html(doc_score_pairs):
    seen, tags = set(), []
    for d, score in doc_score_pairs:
        key = (d.metadata.get('source', '?'), d.metadata.get('page', '?'))
        if key not in seen:
            seen.add(key)
            tags.append(
                f"<span class='source-tag'>📄 {key[0]} — Trang {key[1]}"
                f"<span class='score-badge'>score: {score}</span></span>"
            )
    if not tags:
        return ''
    return f"""
    <div class='sources-wrapper'>
        <div class='sources-label'>📚 Nguồn trích dẫn & Điểm xếp hạng</div>
        {''.join(tags)}
    </div>"""

# ── SIDEBAR_HTML: tĩnh, chunk count cập nhật qua gr.Markdown riêng ───────────
SIDEBAR_HTML = f"""
<div class='stat-card'><div class='label'>Trạng thái</div><div class='value green'>🟢 Sẵn sàng</div></div>
<div class='stat-card'><div class='label'>Mô hình LLM</div><div class='value blue'>gemini-2.5-flash</div></div>
<div class='stat-card'><div class='label'>Embeddings</div><div class='value blue'>gemini-embedding-001</div></div>
"""

def _chunk_count_md():
    """Trả về Markdown hiển thị tổng chunks — dùng để cập nhật động."""
    return f"<div class='stat-card'><div class='label'>Tổng số Chunks</div><div class='value'>{len(all_chunks):,} đoạn</div></div>"

# ── NEW: Hàm xử lý upload PDF ─────────────────────────────────────────────────
def handle_upload_pdfs(files):
    """
    Callback cho gr.File upload.

    Luồng xử lý:
      1. Đọc từng PDF bằng PyPDFLoader
      2. Chia nhỏ bằng RecursiveCharacterTextSplitter (size=1000, overlap=100)
      3. Thêm chunks mới vào vector_db (theo batch 80, delay 60s tránh 429)
      4. Rebuild bm25_retriever từ toàn bộ all_chunks (đảm bảo IDF chính xác)
      5. Trả về (status_message, chunk_count_html) để cập nhật UI
    """
    global bm25_retriever, all_chunks, dense_retriever, sbert_retriever, ABBREV_MAP

    if not files:
        return "⚠️ Chưa chọn file nào.", _chunk_count_md()

    # ── Bước 1 & 2: Load + Chunk ─────────────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    new_chunks  = []   # chunks từ lần upload này
    loaded_info = []   # log mỗi file
    errors      = []

    for file_obj in files:
        file_path = file_obj.name                   # Gradio trả về path tạm
        fname     = os.path.basename(file_path)

        try:
            # Load PDF
            loader = PyPDFLoader(file_path)
            pages  = loader.load()

            if not pages:
                errors.append(f"`{fname}`: không đọc được trang nào")
                continue

            # Làm sạch văn bản (khớp với pipeline Bước 2)
            clean_pages = []
            for doc in pages:
                lines = [ln.strip() for ln in doc.page_content.splitlines() if ln.strip()]
                doc.page_content      = '\n'.join(lines)
                doc.metadata['source'] = fname
                doc.metadata['page']   = doc.metadata.get('page', 0) + 1  # 1-based
                clean_pages.append(doc)

            # Chunk
            file_chunks = splitter.split_documents(clean_pages)
            new_chunks.extend(file_chunks)
            loaded_info.append(f"**{fname}** → {len(file_chunks)} chunks")

        except Exception as e:
            errors.append(f"`{fname}`: {e}")

    if not new_chunks:
        err_str = "\n".join(f"❌ {e}" for e in errors)
        return f"⚠️ Không trích xuất được chunk nào.\n{err_str}", _chunk_count_md()

    # ── Bước 3: Cập nhật Vector DB theo batch (tránh lỗi 429) ────────────────
    BATCH_SIZE    = 80
    total_batches = (len(new_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

    # ── Bước 3 & 4: Rebuild tất cả retrievers từ all_chunks mới ─────────────
    all_chunks.extend(new_chunks)
    bm25_retriever  = BM25Retriever.from_documents(all_chunks)
    bm25_retriever.k = 5
    dense_retriever = TfidfRetriever(chunks=all_chunks, k=5)
    sbert_retriever = SbertDenseRetriever(chunks=all_chunks, k=5)
    ABBREV_MAP      = build_abbrev_map(all_chunks)

    # ── Bước 5: Tạo thông báo kết quả ────────────────────────────────────────
    file_lines = "\n".join(f"  • {info}" for info in loaded_info)
    err_lines  = ("\n" + "\n".join(f"  ⚠️ {e}" for e in errors)) if errors else ""

    status = (
        f"✅ Đã nạp thêm **{len(new_chunks)} chunks** từ {len(loaded_info)} file:\n"
        f"{file_lines}"
        f"{err_lines}\n\n"
        f"📦 Tổng chunks trong hệ thống: **{len(all_chunks):,}**"
    )
    return status, _chunk_count_md()


# ── Streaming respond (giữ nguyên) ────────────────────────────────────────────
def respond_stream(message, chat_history, mode_label, top_k, temp):
    """Generator function cho streaming — yield từng token khi LLM sinh ra."""
    if not message.strip():
        yield '', chat_history, ''
        return

    mode = MODE_LABELS.get(mode_label, 'hybrid')
    llm.temperature = temp

    try:
        doc_scores  = get_retriever_docs(message, mode=mode, k=top_k)
        context_str = format_docs(doc_scores)
        source_html = build_source_html(doc_scores)

        prompt_val = PROMPT.format_messages(context=context_str, question=message)

        partial      = ''
        chat_history = chat_history + [(message, '')]
        for chunk in llm.stream(prompt_val):
            token    = chunk.content if hasattr(chunk, 'content') else str(chunk)
            partial += token
            chat_history[-1] = (message, partial)
            yield '', chat_history, source_html

    except Exception as e:
        chat_history = chat_history + [(message, f'❌ Lỗi: {str(e)}')]
        yield '', chat_history, ''


# ── Gradio Layout ─────────────────────────────────────────────────────────────
with gr.Blocks(
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(primary_hue='blue', secondary_hue='slate',
                         neutral_hue='slate', font=gr.themes.GoogleFont('Inter')),
    title='PTIT AI - RAG System'
) as demo:

    gr.HTML("<h1 class='main-title'> Hệ Thống Hỏi Đáp Tài Liệu</h1>")

    with gr.Row():
        # ── Sidebar ──────────────────────────────────────────────────────────
        with gr.Column(scale=1, min_width=230):
            gr.Markdown('### ⚙️ Cấu hình')
            retriever_radio = gr.Radio(
                choices=list(MODE_LABELS.keys()),
                value='Hybrid RRF(BM25 + SBERT)',
                label='🔍 Phương thức truy xuất',
            )
            top_k_slider = gr.Slider(minimum=1, maximum=10, value=5, step=1,
                                     label='Số đoạn trích dẫn (k)')
            temp_slider  = gr.Slider(minimum=0, maximum=1, value=0.0, step=0.1,
                                     label='Độ sáng tạo (Temperature)')

            gr.Markdown('---')

            # ── NEW: Khu vực tải lên tài liệu ───────────────────────────────
            gr.Markdown('### 📤 Tải lên tài liệu mới')

            upload_file = gr.File(
                file_count='multiple',       # cho phép chọn nhiều file cùng lúc
                file_types=['.pdf'],          # chỉ nhận PDF
                label='Chọn file PDF',
                interactive=True,
            )
            upload_btn = gr.Button(
                '📥 Nạp vào hệ thống',
                variant='secondary',
                size='sm',
            )
            # Hiển thị kết quả xử lý (Markdown để render bold/emoji)
            upload_status = gr.Markdown(
                value='',
                label='Trạng thái upload',
                visible=True,
            )

            gr.Markdown('---')

            # Stat cards tĩnh + chunk count động
            gr.HTML(SIDEBAR_HTML)
            chunk_count_display = gr.HTML(value=_chunk_count_md())  # cập nhật sau mỗi upload

            clear = gr.Button('🗑️ Xóa lịch sử', variant='stop')

        # ── Chat area (giữ nguyên) ────────────────────────────────────────────
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                show_label=False, height=480,
                show_copy_button=True, render_markdown=True,
                latex_delimiters=[
                    {'left': '$$', 'right': '$$', 'display': True},
                    {'left': '$',  'right': '$',  'display': False},
                ],
                elem_classes=['chatbot-wrap']
            )
            sources_display = gr.HTML('', elem_classes=['sources-area'])

            with gr.Row():
                msg = gr.Textbox(
                    placeholder='Nhập câu hỏi về tài liệu của bạn...',
                    show_label=False, scale=9, container=False,
                    elem_classes=['rounded-input']
                )
                submit = gr.Button('Gửi ➤', variant='primary', scale=1,
                                   elem_classes=['send-btn'])

    # ── Event wiring ──────────────────────────────────────────────────────────
    chat_inputs  = [msg, chatbot, retriever_radio, top_k_slider, temp_slider]
    chat_outputs = [msg, chatbot, sources_display]

    msg.submit(respond_stream, chat_inputs, chat_outputs)
    submit.click(respond_stream, chat_inputs, chat_outputs)
    clear.click(lambda: ([], ''), None, [chatbot, sources_display])

    # Upload: nút click → handle_upload_pdfs → cập nhật status + chunk count
    upload_btn.click(
        fn=handle_upload_pdfs,
        inputs=[upload_file],
        outputs=[upload_status, chunk_count_display],
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9617fe29b2d194262e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🔬 BƯỚC 6 (Tuỳ chọn): So sánh BM25 vs Dense vs Hybrid qua terminal

In [ ]:
def compare_retrievers(query, k=3):
    print(f'🔍 Query: "{query}"')
    print('=' * 70)
    for mode, label in [('bm25','📊 BM25'), ('dense','🧠 Dense'), ('hybrid','⚡ Hybrid RRF')]:
        print(f'\n{label} — Top {k} kết quả:')
        for rank, (doc, score) in enumerate(get_retriever_docs(query, mode=mode, k=k), 1):
            src  = doc.metadata.get('source','?')
            page = doc.metadata.get('page','?')
            print(f'  [{rank}] {src} - Trang {page} | score={score}')
            print(f'      {doc.page_content[:120].strip()}...')

test_query = input('Nhập câu hỏi để so sánh: ')
if test_query:
    compare_retrievers(test_query)


Nhập câu hỏi để so sánh: Cách tính điểm MRR (Mean Reciprocal Rank) cho một hệ thống QA?
🔍 Query: "Cách tính điểm MRR (Mean Reciprocal Rank) cho một hệ thống QA?"

📊 BM25 — Top 3 kết quả:
  [1] Chapter 8_ Question Answering.pdf - Trang 46 | score=1.0
      Evaluating Question Answering
3 queries and their ranks of the first relevant documents are:
• Query 1: First relevant r...
  [2] Chapter 8_ Question Answering.pdf - Trang 44 | score=0.5
      Evaluating Question Answering
QA systems give multiple ranked answers, we evaluated by using
mean reciprocal rank, or MR...
  [3] Chapter 8_ Question Answering.pdf - Trang 45 | score=0.3333
      Evaluating Question Answering
Reciprocal Rank (RR):
• For each query, the RR is calculated. If the first relevant result...

🧠 Dense — Top 3 kết quả:
  [1] Chapter 8_ Question Answering.pdf - Trang 46 | score=0.4198
      Evaluating Question Answering
3 queries and their ranks of the first relevant documents are:
• Query 1: First relevant r...
  [2] Cha